# Modelagem da Camada Gold: Fato Estoque

In [ ]:
import sys
from pathlib import Path

# Adiciona o diretório raiz do projeto ao sys.path para importações locais
sys.path.append(str(Path.cwd().parent.parent))

from pyspark.sql import functions as F
from src.modules.spark_session import get_spark_session, close_spark_session
import src.modules.modeling_fato_utils as modeling_fato
import src.modules.utils as utils

In [ ]:
# Inicializa a SparkSession conectada ao cluster do container
spark = get_spark_session("ModelagemGoldFatoEstoque")

# Leitura das tabelas de origem

In [ ]:
# Define caminhos das origens na Silver e Gold
silver_estoque_path = "s3a://silver/estoque"
gold_dim_fornecedor_path = "s3a://gold/dim_fornecedor"

# Lê os dados definindo como None caso a origem não exista
try:
    df_estoque = spark.read.parquet(silver_estoque_path)
except Exception as e:
    print(f"Aviso: Tabela Silver de Estoque não encontrada: {e}")
    df_estoque = None

try:
    df_dim_fornecedor = spark.read.parquet(gold_dim_fornecedor_path)
except Exception as e:
    print(f"Aviso: Tabela Gold dim_fornecedor não encontrada: {e}")
    df_dim_fornecedor = None

# Visualização Prévia dos Dados de Entrada

In [ ]:
if df_estoque is not None:
    print("=== Colunas em Silver Estoque ===")
    display(df_estoque.limit(5).toPandas())

if df_dim_fornecedor is not None:
    print("=== Colunas em Gold dim_fornecedor ===")
    display(df_dim_fornecedor.limit(5).toPandas())

# Cria a Fato Estoque (fato_estoque)

In [ ]:
# Executa a lógica de modelagem da fato
df_fato_estoque = modeling_fato.create_fato_estoque(df_estoque, df_dim_fornecedor)

if df_fato_estoque is not None:
    # Adiciona a data de carga
    df_fato_estoque = df_fato_estoque.withColumn("data_carga", F.to_date(F.lit(utils.get_current_date_str())))
    
    # Exibe informações sobre o DataFrame gerado
    print(f"Quantidade total de registros na fato estoque: {df_fato_estoque.count()}")
    df_fato_estoque.printSchema()
    display(df_fato_estoque.limit(10).toPandas())
else:
    print("Nenhuma fato de estoque foi processada (tabela Silver de estoque estava ausente).")

In [ ]:
# Finaliza a sessão do Spark
close_spark_session(spark)